# Lab en Clase: Decision Models con la Decisions API

Muchas tareas con LLMs en empresa **no necesitan generar texto**: necesitan **elegir** entre opciones cerradas.
¿A qué departamento va este ticket? ¿Es urgente? ¿El cliente amenaza con irse?

Hasta ahora lo resolvíamos con una llamada generativa (prompt + salida estructurada). La **Decisions API** de OpenAI (sobre `gpt-6-luna`) está pensada solo para esto: le haces preguntas cerradas y devuelve la opción elegida **con su probabilidad**, mucho más rápido.

### Qué practicarás
- Triaje de tickets con una llamada generativa (`responses.parse` + Pydantic)
- El mismo triaje con la Decisions API: preguntas `choice`, `score` y `predicate`
- Comparar las dos vías como un data scientist: **acierto, latencia y coste**
- Usar la **confianza** para decidir qué se automatiza y qué revisa una persona

### 🧭 Antes de empezar

- **Cómo se usa:** ejecuta las celdas en orden, de arriba abajo, con `Mayús + Enter`. Si una celda da error, lee la última línea: casi siempre dice qué falta.
- **Si algo se queda raro:** menú *Entorno de ejecución → Reiniciar y ejecutar todo*.
- **Tu API key:** en Colab, guárdala como secreto `OPENAI_API_KEY` (icono de la llave, a la izquierda). Nunca la pegues dentro de una celda.

**Vocabulario de este notebook**

| Término | Qué es |
|---|---|
| **Decision model** | un modelo que no escribe texto: elige una opción y dice con qué probabilidad. |
| **`predicate` · `choice` · `score`** | pregunta de sí/no, elegir de una lista y puntuar en una escala. |
| **Confianza** | la probabilidad que el modelo da a su elección. |
| **Umbral** | el mínimo de confianza para automatizar; por debajo, el caso va a una persona. |


## Sección 1: Configuración

In [ ]:
!pip install -q --upgrade openai pandas matplotlib

In [ ]:
import os, getpass, time, json
import pandas as pd
import matplotlib.pyplot as plt
from typing import Literal
from pydantic import BaseModel
from openai import OpenAI

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key: ")
client = OpenAI(api_key=OPENAI_API_KEY)

MODELO = "gpt-6-luna"

# Precios por millón de tokens (USD). Compruébalos en https://openai.com/api/pricing antes de sacar conclusiones
PRECIO_LUNA_INPUT = 0.10
PRECIO_LUNA_OUTPUT = 0.50
PRECIO_DECISIONS_INPUT = 0.10   # en la Decisions API el output no se cobra

## Sección 2: El dataset: tickets de soporte

Una operadora ficticia de fibra y móvil, **LumenFibra**, recibe tickets de clientes. Cada ticket tiene etiquetas puestas a mano (nuestra *verdad terreno*):
- `departamento`: facturacion · tecnico · bajas · comercial
- `urgencia`: baja · media · alta
- `riesgo_baja`: ¿el cliente amenaza con irse?

In [ ]:
tickets = [
    ("Me habéis cobrado dos veces la factura de octubre, quiero que me devolváis el importe ya.", "facturacion", "alta", False),
    ("¿Podéis enviarme un duplicado de la factura de septiembre? La necesito para la empresa.", "facturacion", "baja", False),
    ("Llevo tres días sin internet y teletrabajo. Si mañana no está arreglado me voy a Movistar.", "tecnico", "alta", True),
    ("El wifi va lento en la habitación del fondo, ¿algún consejo para mejorar la cobertura?", "tecnico", "baja", False),
    ("Quiero darme de baja de la línea móvil adicional, ya no la uso.", "bajas", "media", False),
    ("¿Tenéis alguna oferta para añadir televisión a mi fibra?", "comercial", "baja", False),
    ("Me ha subido la cuota 8 euros sin avisar. O me lo explicáis o cambio de compañía.", "facturacion", "media", True),
    ("El router parpadea en rojo desde esta mañana y no hay conexión en toda la casa.", "tecnico", "alta", False),
    ("Me mudo el mes que viene, ¿cómo traslado la fibra a la nueva dirección?", "comercial", "media", False),
    ("Solicito la baja total del servicio y la portabilidad de mi número a otra operadora.", "bajas", "alta", True),
    ("No entiendo el concepto 'servicios premium' de mi última factura, yo no lo he contratado.", "facturacion", "media", False),
    ("¿Cuánto cuesta subir de 600 Mb a 1 Gb?", "comercial", "baja", False),
    ("Se corta la llamada cada vez que paso del salón a la cocina, el móvil pierde cobertura en casa.", "tecnico", "media", False),
    ("Llevo 10 años con vosotros y otra compañía me ofrece la mitad. ¿Me mejoráis el precio o me voy?", "comercial", "media", True),
    ("Quiero cancelar el seguro del móvil que contraté con el terminal.", "bajas", "baja", False),
    ("Me han cargado la penalización de permanencia pero ya había cumplido los 12 meses.", "facturacion", "alta", False),
    ("La fibra se cae todas las noches sobre las 23:00 durante unos minutos.", "tecnico", "media", False),
    ("¿Me podéis mandar un técnico? El cable de la fibra está pelado y me da miedo.", "tecnico", "alta", False),
    ("Me interesa el pack familiar con 4 líneas, ¿qué condiciones tiene?", "comercial", "baja", False),
    ("Estoy harto: tercera avería este mes. Tramitad mi baja.", "bajas", "alta", True),
    ("¿Puedo cambiar la fecha de cargo del recibo al día 5?", "facturacion", "baja", False),
    ("Después de la actualización del router ya no me funciona la impresora por wifi.", "tecnico", "baja", False),
    ("Quiero contratar una segunda línea para mi hija.", "comercial", "baja", False),
    ("Nadie me da solución a la factura duplicada, llevo dos semanas llamando. Último aviso antes de reclamar a consumo.", "facturacion", "alta", True),
]

df = pd.DataFrame(tickets, columns=["texto", "departamento", "urgencia", "riesgo_baja"])
print(f"{len(df)} tickets")
df.head()

## Sección 3: Vía A: llamada generativa con salida estructurada

Lo que ya sabemos hacer: un prompt, un esquema Pydantic y `responses.parse`. Usamos `effort="none"` porque clasificar no requiere razonar (es la opción generativa más rápida).

In [ ]:
class Triaje(BaseModel):
    departamento: Literal["facturacion", "tecnico", "bajas", "comercial"]
    urgencia: Literal["baja", "media", "alta"]
    riesgo_baja: bool

INSTRUCCIONES = (
    "Eres el sistema de triaje de soporte de una operadora de fibra y móvil. "
    "departamento: facturacion (cobros, facturas, importes), tecnico (averías, cobertura, equipos), "
    "bajas (cancelar servicios o portabilidad), comercial (contratar, ofertas, cambios de tarifa, traslados). "
    "urgencia: alta si el cliente está sin servicio, hay dinero cobrado de más o un riesgo; baja si es una consulta. "
    "riesgo_baja: true si el cliente amenaza con irse o pide la baja/portabilidad."
)

def triaje_generativo(texto):
    t0 = time.time()
    r = client.responses.parse(
        model=MODELO,
        reasoning={"effort": "none"},
        instructions=INSTRUCCIONES,
        input=texto,
        text_format=Triaje,
    )
    return r.output_parsed, time.time() - t0, r.usage

res_a = []
for texto in df["texto"]:
    pred, seg, u = triaje_generativo(texto)
    res_a.append({**pred.model_dump(), "latencia_s": seg, "tok_in": u.input_tokens, "tok_out": u.output_tokens})

gen = pd.DataFrame(res_a)
gen.head()

## Sección 4: Vía B: la Decisions API

Una sola llamada por ticket con **tres preguntas**, una de cada tipo:

| Tipo | Para qué | Devuelve |
|---|---|---|
| `choice` | elegir una opción de una lista | `choice`, `confidence` y la probabilidad de cada opción |
| `score` | una escala ordenada (2 a 10 niveles) | `score` (media ponderada de los niveles) y la probabilidad de cada nivel |
| `predicate` | una pregunta de sí/no | `probability` de que sea verdad |

In [ ]:
PREGUNTAS = [
    {
        "type": "choice",
        "name": "departamento",
        "instructions": "¿Qué departamento debe atender este ticket?",
        "choices": [
            {"value": "facturacion", "description": "Cobros, facturas, importes, recibos, penalizaciones"},
            {"value": "tecnico", "description": "Averías, cortes, cobertura, router, equipos"},
            {"value": "bajas", "description": "Cancelar un servicio, baja total o portabilidad a otra operadora"},
            {"value": "comercial", "description": "Contratar, ofertas, cambios de tarifa, traslados, retención por precio"},
        ],
    },
    {
        "type": "score",
        "name": "urgencia",
        "instructions": "¿Con qué urgencia hay que atender este ticket?",
        "levels": [
            {"label": "baja", "description": "Consulta o gestión sin impacto inmediato"},
            {"label": "media", "description": "Molestia o problema parcial; hay que resolverlo en días"},
            {"label": "alta", "description": "Sin servicio, dinero cobrado de más, riesgo físico o cliente al límite"},
        ],
    },
    {
        "type": "predicate",
        "name": "riesgo_baja",
        "instructions": "¿El cliente amenaza con irse a otra compañía o pide la baja o la portabilidad?",
    },
]

# Probamos con un ticket
d = client.decisions.create(model=MODELO, input=df.loc[2, "texto"], questions=PREGUNTAS)
print(df.loc[2, "texto"], "\n")
for a in d.answers:
    print(a.model_dump_json(indent=2))

Fíjate en lo que no tenemos con la vía generativa: **la probabilidad de cada opción**. No es solo «técnico», es «técnico con un 97%». Esa cifra es oro para decidir cuándo fiarse del modelo (Sección 6).

Ahora, todos los tickets:

In [ ]:
NIVELES = ["baja", "media", "alta"]

def triaje_decisions(texto):
    t0 = time.time()
    d = client.decisions.create(model=MODELO, input=texto, questions=PREGUNTAS)
    seg = time.time() - t0
    r = {a.name: a for a in d.answers}
    rechazos = [a.name for a in d.answers if a.type == "refusal"]
    if rechazos:   # el modelo puede negarse a responder alguna pregunta (contenido inapropiado, etc.)
        raise ValueError(f"Pregunta(s) rechazada(s): {rechazos}")
    dep, urg, rb = r["departamento"], r["urgencia"], r["riesgo_baja"]
    return {
        "departamento": dep.choice,
        "conf_departamento": dep.confidence,
        "urgencia": NIVELES[round(urg.score)],     # score es la media ponderada de los índices 0..2
        "score_urgencia": urg.score,
        "riesgo_baja": rb.probability >= 0.5,
        "prob_riesgo_baja": rb.probability,
        "latencia_s": seg,
        "tok_in": d.usage.input_tokens if d.usage else None,
    }

dec = pd.DataFrame([triaje_decisions(t) for t in df["texto"]])
dec.head()

## Sección 5: Comparativa: acierto, latencia y coste

In [ ]:
def resumen(pred, nombre, precio_in, precio_out=0.0):
    acierto = {c: (pred[c] == df[c]).mean() for c in ["departamento", "urgencia", "riesgo_baja"]}
    coste = (pred["tok_in"].sum() * precio_in + pred.get("tok_out", pd.Series([0])).sum() * precio_out) / 1e6
    return {
        "vía": nombre,
        **{f"acierto_{k}": f"{v:.0%}" for k, v in acierto.items()},
        "latencia_media_s": round(pred["latencia_s"].mean(), 2),
        "latencia_p95_s": round(pred["latencia_s"].quantile(0.95), 2),
        "coste_1000_tickets_usd": round(coste / len(pred) * 1000, 4),
    }

pd.DataFrame([
    resumen(gen, "A · generativa (parse)", PRECIO_LUNA_INPUT, PRECIO_LUNA_OUTPUT),
    resumen(dec, "B · Decisions API", PRECIO_DECISIONS_INPUT),
]).set_index("vía").T

**Qué mirar:**
- **Acierto**: con 24 tickets, una diferencia de 1 ticket son 4 puntos. No saques conclusiones fuertes con tan pocos datos (en un proyecto real: cientos de ejemplos etiquetados).
- **Latencia**: es donde la Decisions API marca la diferencia (OpenAI anuncia ~150 ms frente a ~1,5 s de una llamada normal de Luna).
- **Coste**: la Decisions API no cobra el output, pero **las preguntas (con sus descripciones) cuentan como input** en cada llamada. Con tickets cortos y preguntas largas puede salir *más cara* que la vía generativa; con textos largos y muchas preguntas en la misma llamada, más barata. No te fíes del marketing: mídelo con tus datos, como acabamos de hacer.

**¿Dónde fallan?** Revisemos los desacuerdos con la verdad terreno:

In [ ]:
comp = df[["texto", "departamento"]].copy()
comp["generativa"] = gen["departamento"]
comp["decisions"] = dec["departamento"]
comp["confianza"] = dec["conf_departamento"].round(2)
comp[(comp["generativa"] != comp["departamento"]) | (comp["decisions"] != comp["departamento"])]

Muchos errores son **casos ambiguos** también para una persona (¿«me mejoráis el precio o me voy» es comercial o bajas?). Fíjate en si la confianza de esos casos es más baja que la del resto.

## Sección 6: Usar la confianza: qué automatizar y qué revisa una persona

En producción no hace falta automatizarlo todo. Una estrategia típica: si la confianza supera un umbral, el ticket se enruta solo; si no, va a una persona. ¿Dónde ponemos el umbral?

In [ ]:
filas = []
for umbral in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    auto = dec["conf_departamento"] >= umbral
    acierto_auto = (dec.loc[auto, "departamento"] == df.loc[auto, "departamento"]).mean() if auto.any() else float("nan")
    filas.append({"umbral": umbral, "% automatizado": auto.mean(), "acierto en lo automatizado": acierto_auto})

curva = pd.DataFrame(filas)
print(curva.to_string(index=False, formatters={"% automatizado": "{:.0%}".format, "acierto en lo automatizado": "{:.0%}".format}))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(curva["% automatizado"], curva["acierto en lo automatizado"], marker="o")
for _, f in curva.iterrows():
    ax.annotate(f"{f['umbral']:.2f}", (f["% automatizado"], f["acierto en lo automatizado"]),
                textcoords="offset points", xytext=(5, 5), fontsize=8)
ax.set_xlabel("% de tickets que se enrutan solos")
ax.set_ylabel("acierto en esos tickets")
ax.set_title("Cobertura vs acierto según el umbral de confianza")
ax.grid(alpha=0.3)
plt.show()

Es el mismo razonamiento que una curva precisión-cobertura en machine learning clásico: **subir el umbral aumenta el acierto pero reduce lo que se automatiza**. La decisión del umbral es de negocio (¿cuánto cuesta un ticket mal enrutado frente a uno revisado a mano?), no técnica.

> La vía generativa no da este control: devuelve una etiqueta sin probabilidad. Puedes pedirle una «confianza» en el JSON, pero es un número que el modelo escribe, no una probabilidad real.

## Sección 7: Ejercicio

1. Añade una cuarta pregunta `predicate` llamada `requiere_tecnico`: «¿Hace falta enviar un técnico al domicilio?». Recuerda: van todas en la misma llamada (hasta 400 preguntas por petición).
2. Cambia la pregunta de urgencia a 5 niveles. ¿Mejora o empeora el acierto? ¿Por qué crees que pasa?
3. Busca el umbral de confianza con el que se automatiza al menos el 70% de los tickets. ¿Qué acierto tienes en ese punto?

## Para saber más: decision models

- **Decisions API (OpenAI)**: `POST /v1/decisions`, solo `gpt-6-luna`. Acepta texto, imágenes y JSON como entrada. Preguntas `predicate`, `choice` y `score`. Solo se cobra el input.
- **Jev (TypeSafe)**: un modelo de decisión **no generativo** que nació antes y al que la Decisions API responde. Mismo concepto (respuestas cerradas con probabilidad), API distinta. No lo usamos en el lab, pero en las diapositivas comparamos los dos.
- **Cuándo usar cada cosa:**
  - Respuesta cerrada, mucho volumen, latencia crítica → *decision model*
  - Necesitas texto, explicación o extraer datos abiertos → llamada generativa con salida estructurada
  - Problema de varios pasos con herramientas → agente (Clases 3 y 4)